# Workspace RayCluster + job client — Ray Train + MLflow

Same pattern as Topics 1–2: `Cluster` + `job_client`, with the workshop **2×GPU** cluster shape.

Runs `train_fashion_mnist.py` — Ray Train `TorchTrainer` on FashionMNIST, then logs params/metrics and registers a PyTorch model in **MLflow** (OpenShift AI).

`ScalingConfig(num_workers=2, use_gpu=True)` must match the RayCluster worker/GPU count.

Workers need network egress to download FashionMNIST and to reach the in-cluster MLflow service.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from workshop_bootstrap import setup_workshop_path

REPO_ROOT = setup_workshop_path()
print(f"Repo root: {REPO_ROOT}")

## Authenticate

Same credentials as Topic 1 — OpenShift Console → **Copy login command** → Display token.


In [ ]:
from kube_authkit import AuthConfig, get_k8s_client
from codeflare_sdk import set_api_client, list_local_queues, view_clusters

OPENSHIFT_SERVER = "https://api.YOUR_CLUSTER:6443"
OPENSHIFT_TOKEN = "REPLACE_WITH_YOUR_TOKEN"

auth_config = AuthConfig(
    method="openshift",
    k8s_api_host=OPENSHIFT_SERVER.strip(),
    token=OPENSHIFT_TOKEN.strip(),
    verify_ssl=False,
)
set_api_client(get_k8s_client(config=auth_config))

NAMESPACE = "ray-workshop"
LOCAL_QUEUE = "ray-workshop-queue"

list_local_queues(NAMESPACE)

## MLflow tracking URI

OpenShift AI 3.4 managed MLflow (not the old `mlflow-server` in a `mlflow` project):

```sh
oc get mlflow mlflow -n redhat-ods-applications -o jsonpath='{.status.address.url}{"\n"}'
```

UI (browser):

```sh
oc get mlflow mlflow -n redhat-ods-applications -o jsonpath='{.status.url}{"\n"}'
```

Use the **address URL** (HTTPS :8443) as `MLFLOW_TRACKING_URI`. Set `MLFLOW_TRACKING_AUTH=kubernetes-namespaced`. If TLS verify fails against the service cert, also set `MLFLOW_TRACKING_INSECURE_TLS=true` in the job env.


In [ ]:
# OpenShift AI managed MLflow (in-cluster tracking API)
# Discover: oc get mlflow mlflow -n redhat-ods-applications -o jsonpath='{.status.address.url}{"\n"}'
MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc:8443"
MLFLOW_EXPERIMENT = "ray-workshop-fashion-mnist"
MLFLOW_REGISTERED_MODEL = "ray-workshop-fashion-mnist"

print("MLFLOW_TRACKING_URI:", MLFLOW_TRACKING_URI)


## Create the RayCluster

Two workers with 1 GPU each (workshop default via `workshop_cluster_configuration`).

In [ ]:
from codeflare_sdk import Cluster
from workshop_bootstrap import workshop_cluster_configuration

cluster = Cluster(
    workshop_cluster_configuration(
        name="ray-workshop-train",
        namespace=NAMESPACE,
        local_queue=LOCAL_QUEUE,
    )
)

cluster.apply()
cluster.wait_ready()
cluster.details()

## Submit Ray Train job

Installs `torch` / `torchvision` / `mlflow` via `runtime_env.pip` when needed. Prefer a CUDA-capable Ray image from [Supported Configurations](https://access.redhat.com/articles/6856871).

MLflow env vars are passed into the job so the Ray cluster can reach your tracking server.


In [ ]:
import time

client = cluster.job_client

submission_id = client.submit_job(
    entrypoint="python extras/scripts/train_fashion_mnist.py",
    runtime_env={
        "working_dir": str(REPO_ROOT),
        "pip": ["torch", "torchvision", "mlflow"],
        "env_vars": {
            "NUM_EPOCHS": "3",
            "NUM_WORKERS": "2",
            "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
            "MLFLOW_TRACKING_AUTH": "kubernetes-namespaced",
            "MLFLOW_TRACKING_INSECURE_TLS": "true",
            "MLFLOW_EXPERIMENT": MLFLOW_EXPERIMENT,
            "MLFLOW_REGISTERED_MODEL": MLFLOW_REGISTERED_MODEL,
            "RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S": "120",
        },
    },
)
print(f"Submitted: {submission_id}")

terminal = {"SUCCEEDED", "FAILED", "STOPPED"}
deadline = time.time() + 1800

while time.time() < deadline:
    status = client.get_job_status(submission_id)
    print(f"Job {submission_id}: {status}")
    if str(status).split(".")[-1].upper() in terminal:
        break
    time.sleep(20)
else:
    raise TimeoutError(f"Timed out waiting for job {submission_id}")

print(client.get_job_logs(submission_id))


## Observe

1. Ray Dashboard from `view_clusters()` → **Jobs** for training progress.
2. MLflow UI → experiment **`ray-workshop-fashion-mnist`** → run metrics + registered model **`ray-workshop-fashion-mnist`**.


In [ ]:
view_clusters(NAMESPACE)

## Tear down

In [ ]:
cluster.down()
print("Cluster down.")

## Verify

1. Job reaches `SUCCEEDED`.
2. Logs show epoch losses, an `MLflow run_id=...` line, and `Done. Ray Train FashionMNIST finished successfully.`
3. MLflow UI shows the experiment run and registered model `ray-workshop-fashion-mnist`.
4. If the job stays PENDING, check that `NUM_WORKERS` matches GPU workers (workshop: 2).
